# G1 Academy Bonus - Task 16: SLAM Pickup and Delivery


## Introduction
Map a space, save two named points, then have the robot fetch an object from one and deliver it to the other -- reusing `add_point`/`remove_point` from Day 2's SLAM task, plus a saved arm pose and the Dex3 hand.

**Safety:**
- Clear the space before `start_mapping`/driving and before any navigation call.
- `g1.interpolate_to_pose(...)` already enforces a per-joint speed cap (`DEFAULT_MAX_JOINT_SPEED_RAD_S = 0.6` rad/s) and a hard damping cap (`MAX_KD = 10.0`) -- you do not set these yourself, but know they exist.
- Keep hands clear of the Dex3 hand while it opens/closes.


In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)
g1.damp_mode(); g1.prepare_mode(); g1.walk_mode()


## Task 1 - map the space: start_mapping / stop_mapping
Drive the robot around between these two calls (e.g. `g1.loco_move(0.15, 0, 0.4, duration_s=8)`, or Day 2's `walk_u_path`) so the LiDAR sees enough of the room.


In [ ]:
g1.start_mapping(slam_type="indoor")


In [ ]:
g1.stop_mapping(save_path="my_map")


## Task 2 - relocalize: relocate


In [ ]:
g1.relocate()


## Named points: add_point / remove_point
Same small helper you built in Day 2's SLAM task -- a JSON dict of `{name: [x, y, yaw]}` on disk, keyed off `g1.get_slam_pose()`, which is what `g1.navigate_to_point(name)` reads.


In [ ]:
import json
from pathlib import Path

POINTS_PATH = "slam_points.json"

def add_point(g1, name, points_path=POINTS_PATH):
    path = Path(points_path)
    points = json.loads(path.read_text()) if path.exists() else {}
    points[name] = list(g1.get_slam_pose())
    path.write_text(json.dumps(points))
    return points[name]

def remove_point(name, points_path=POINTS_PATH):
    path = Path(points_path)
    points = json.loads(path.read_text()) if path.exists() else {}
    points.pop(name, None)
    path.write_text(json.dumps(points))


Walk to the object's location and to the delivery location (e.g. with `g1.loco_move(...)`), saving a named point at each:


In [ ]:
add_point(g1, "pickup")
add_point(g1, "dropoff")


## Task 3 - grab and deliver
A saved right-arm-forward pose is provided in `right_arm_forward_pose.json` (a flat `{joint_id: q}` dict, ready for `g1.interpolate_to_pose(...)`) -- reach for the object with it, grip with `g1.close_dex3_hand()`, deliver with `g1.navigate_to_point(...)`, then release with `g1.open_dex3_hand()`.


In [ ]:
g1.navigate_to_point("pickup")


In [ ]:
g1.open_dex3_hand()


In [ ]:
import json
pose = json.loads(open("right_arm_forward_pose.json").read())
g1.interpolate_to_pose(pose)


In [ ]:
g1.close_dex3_hand()


In [ ]:
g1.navigate_to_point("dropoff")


In [ ]:
g1.open_dex3_hand()
